setfit prefers:

 pip install "torch==2.5.1"
  "transformers==4.48.*" "sentence-
  transformers==5.0.0" "setfit==1.1.3" datasets
  evaluate scikit-learn

In [1]:
import torch
print(torch.__version__)
print(torch.__file__)

if hasattr(torch, "xpu"):
    torch.xpu.empty_cache = lambda: None

device = torch.device("cpu")
torch.set_default_dtype(torch.float32)
print(f"Using {device}")

2.5.1+cpu
/home/urdatorn/.pyenv/versions/setfit-cpu/lib/python3.11/site-packages/torch/__init__.py
Using cpu


In [2]:
import transformers.training_args as training_args

/home/urdatorn/.pyenv/versions/setfit-cpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sentence_transformers import SentenceTransformer
from setfit import SetFitModel

body = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=str(device))
model = SetFitModel(model_body=body)

In [4]:
from datasets import load_dataset

dataset = load_dataset("DigPhil/sentiment-grc")

train_dataset = dataset["train"]
val_dataset = dataset["validation"]
test_dataset = dataset["test"]

Batch size 32 crashar på min dator, antagligen för att RAM tar slut.

In [ ]:
from setfit import TrainingArguments

args = TrainingArguments(
    batch_size=8,
    num_epochs=1,
    max_steps=-1,
    body_learning_rate=(2e-5, 1e-5)
    head_learning_rate=1e-2
)

In [6]:
from setfit import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

Setfit tränar genom att ta par av texter och fråga modellen 

Så träningsstorleken blir (n, k) = n!/k!(n-k)! = 578 * 577 ≈ 170k.

In [ ]:
trainer.train()

***** Running training *****
  Num unique pairs = 218250
  Batch size = 8
  Num epochs = 1


Step,Training Loss
1,0.165600
50,0.337000
100,0.298500
150,0.267700
200,0.258100


In [ ]:
trainer.evaluate(test_dataset)

In [ ]:
preds = model.predict([
    "ὦ μιαρέ"
])